In [16]:
import os
os.environ['SPARK_LOCAL_IP']='127.0.0.1'

from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [26]:
from pyspark.sql.functions import col, sum as spark_sum, avg, when

In [17]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")


Dataset dibuat: 1000 baris


In [18]:
# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /tarin/home/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /tarin/home/tugas4/
print("Berhasil diunggah ke HDFS: /tarin/home/tugas4/transaksi_september_2026.csv")

Berhasil diunggah ke HDFS: /tarin/home/tugas4/transaksi_september_2026.csv


In [19]:
df = spark.read.csv(
    "hdfs://localhost:9000/tarin/home/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)

df.printSchema()
print("Jumlah baris:", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

In [21]:
jumlah_kosong = df.filter(df["rating"].isNull()).count()
print("Jumlah rating kosong:", jumlah_kosong)

df = df.na.drop(subset=["rating"])
print("Jumlah baris setelah drop:", df.count())

Jumlah rating kosong: 204
Jumlah baris setelah drop: 796


In [28]:
df = df.withColumn("total_pendapatan", df["unit_terjual"] * df["harga_satuan"])
df = df.withColumn(
    "tier_transaksi",
    when(df["total_pendapatan"] > 500000, "Besar").otherwise("Kecil")
)
df.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
|ORD-3010|           4|      350000|         1400000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [35]:
# D1. Kategori dengan total_pendapatan tertinggi
df.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
).orderBy(col("total_pendapatan").desc()).show()

# D2. Kota dengan jumlah transaksi tier "Besar" terbanyak
df.filter(col("tier_transaksi") == "Besar") \
  .groupBy("kota") \
  .count() \
  .orderBy(col("count").desc()) \
  .show()

# D3. Rata-rata rating per metode_pembayaran
df.groupBy("metode_pembayaran").agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(col("rata_rata_rating").desc()).show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       108285000|
|   Makanan & Minuman|       106085000|
|             Fashion|       101725000|
|Kesehatan & Kecan...|        98040000|
|            Olahraga|        94230000|
|          Elektronik|        89325000|
+--------------------+----------------+

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   74|
|   Kebumen|   63|
|Yogyakarta|   61|
| Purworejo|   55|
|  Magelang|   52|
|  Semarang|   47|
+----------+-----+

+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|4.172413793103448|
|    Transfer Bank| 4.16256157635468|
|         E-Wallet|4.135678391959799|
|     Kartu Kredit|4.109947643979058|
+-----------------+-----------------+



In [37]:
df.write.mode("overwrite").option("header", True).csv(
    "hdfs://localhost:9000/tarin/home/tugas4/hasil_transformasi"
)
print("Berhasil disimpan ke HDFS.")

Berhasil disimpan ke HDFS.


In [38]:
!hdfs dfs -ls /tarin/home/tugas4/hasil_transformasi

Found 2 items
-rw-r--r--   3 tarin supergroup          0 2026-09-10 19:07 /tarin/home/tugas4/hasil_transformasi/_SUCCESS
-rw-r--r--   3 tarin supergroup      77337 2026-09-10 19:07 /tarin/home/tugas4/hasil_transformasi/part-00000-384e7057-4e48-480e-b9e0-22c57630bef9-c000.csv
